In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType
from pyspark.sql.types import IntegerType, DecimalType, StringType, FloatType, StructType, StructField, DoubleType, LongType  # Import necessary types
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when
from datetime import date
from pyspark.sql import DataFrame
from functools import reduce


In [0]:
Source = "Adgo"
Domain = "Policy"

In [0]:
def readData():
 
  bvPath = "abfss://bv-commercial@zukien1prdaladlsg204.dfs.core.windows.net/data/"
  businessViewDict = {
  "Contract" : bvPath + "AdGo/contract/"
  ,"Schedule" : bvPath + "AdGo/schedule/"
  ,"Party" : bvPath + "AdGo/party/"
  ,"Party Org" : bvPath + "AdGo/partyorganisation/"
  ,"pipolicy" : bvPath + "AdGo/Policy/pipolicy" 
  ,"policy" : bvPath + "AdGo/Policy/policy"
  ,"policy_mta" : bvPath + "AdGo/Policy/policy_mta"
  ,"policyendorsement" : bvPath + "AdGo/Policy/policyendorsement"
  ,"policypartydetails" : bvPath + "AdGo/Policy/policypartydetails"
  ,"ppppolicy" : bvPath + "AdGo/Policy/ppppolicy"
  ,"contractlicensedetailshistory" : bvPath + "AdGo/contractlicensedetailshistory/"
  ,"contractpartydetails" : bvPath + "AdGo/contractpartydetails/"
  ,"lnperilscores" : bvPath + "AdGo/lnperilscores/"
  ,"scheduleextensiondata" : bvPath + "AdGo/scheduleextensiondata" # no data
  ,"partyorganisationinsured" : bvPath + "AdGo/partyorganisationinsured/"
  ,"perilscores" : bvPath + "AdGo/perilscores/" # no data
  ,"picontract" : bvPath + "AdGo/picontract"
  ,"pppcontract" : bvPath + "AdGo/pppcontract"
  ,"scheduleperilscores" : bvPath + "AdGo/scheduleperilscores"
  ,"average_rate_per_vehicle" : bvPath + "AdGo/average_rate_per_vehicle"
  ,"profit_loss_ratio_up_to_date" : bvPath + "AdGo/loss_ratio_reporting/profit_loss_ratio_up_to_date"
  ,"profit_loss_ratio_ten_mon" : bvPath + "AdGo/loss_ratio_reporting/profit_loss_ratio_ten_mon"
  }

  spark_df = {}
  for df,path in businessViewDict.items():
    spark_df[df]=spark.read.format("delta").load(path)

  try:
    FilterRulesDF = spark.read.option('header',True).csv("abfss://bv-commercial@zukien1prdaladlsg205.dfs.core.windows.net/data/data_quality/Execution_Files/Source/Adgo/FilterDataCondition_Adgo020924.csv")
    return spark_df,FilterRulesDF
  except:
    return spark_df

  File <command-3893387706113876>, line 1
    "abfss://bv-commercial@zukien1prdaladlsg204.dfs.core.windows.net/data/AdGo/schedule/"def readData():
                                                                                         ^
SyntaxError: invalid syntax


In [0]:
def filterData(spark_df,FilterRulesDF):
    if FilterRulesDF.count()>=0:
        for i in FilterRulesDF.collect():
          condition = f"{i['Variable']} {i['Condition']} {i['Criterion']}".replace('[','').replace(']','')
          spark_df[i['Origin']] = spark_df[i['Origin']].filter(condition)

    else:
           print('No filtering required') 
    return spark_df

In [0]:
# Coverts the datatype of date and timestamp columns to date 

# Set legacy time parser policy
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

def convert_date_columns(spark_df):
    for df_name, df in spark_df.items():
        columns = df.columns
        
        for col in columns:
            if 'date' in col.lower():
                try:
                    # Convert the column to DateType
                    df = df.withColumn(col, F.to_date(F.col(col), 'yyyy-MM-dd'))
                    
                    # Check if the conversion was successful
                    if df.filter(F.col(col).isNull()).count() > 0:
                        print(f"Column '{col}' in DataFrame '{df_name}' contains invalid date values.")
                    else:  
                        print(f"Column '{col}' in DataFrame '{df_name}' converted to DateType.")
                except Exception as e:
                    print(f"Error converting column '{col}' in DataFrame '{df_name}': {e}")
        
        # Update the DataFrame in the dictionary
        spark_df[df_name] = df

    return spark_df


In [0]:
class EDA:  # EDA for Input data

    def __init__(self, spark_session):
        self.spark = spark_session  # Store the Spark session

        # Define the schema for the date summary DataFrame
        self.schema = StructType([
            StructField("Table", StringType(), True),
            StructField("Column", StringType(), True),
            StructField("Year", IntegerType(), True),
            StructField("Month", IntegerType(), True),
            StructField("MonthlyCount", LongType(), True),
            StructField("MinDate", DateType(), True),
            StructField("MaxDate", DateType(), True),
            StructField("UniqueDayCount", LongType(), True)
        ])

        # Define the schema for the numeric summary DataFrame
        self.schema_num = StructType([
            StructField("Table", StringType(), True),
            StructField("Column", StringType(), True),
            StructField("Count", LongType(), True),
            StructField("Mean", DoubleType(), True),
            StructField("Median", DoubleType(), True),
            StructField("Mode", DoubleType(), True),
            StructField("Standard_Deviation", DoubleType(), True),
            StructField("Variance", DoubleType(), True),
            StructField("Min", DoubleType(), True),
            StructField("Max", DoubleType(), True),
            StructField("Range", DoubleType(), True),
            StructField("Unique_Count", LongType(), True),
            StructField("Missing_Count", LongType(), True),
            StructField("Missing_Percentage", DoubleType(), True),
            StructField("Percentile_25", DoubleType(), True),
            StructField("Percentile_50", DoubleType(), True),
            StructField("Percentile_75", DoubleType(), True)
        ])

        # Define the schema for the categorical DataFrame
        self.schema_cat = StructType([
            StructField("Table_Name", StringType(), True),
            StructField("Column_Name", StringType(), True),
            StructField("Value", StringType(), True),
            StructField("Condition", StringType(), True),
            StructField("Value_count", LongType(), True),  # Use LongType for bigint
            StructField("Percentage", DoubleType(), True)  # Use DoubleType for double
        ])

    def global_df(self, df_name, df):
        # Create a row for the DataFrame's name, shape, and column count
        return (df_name, df.count(), len(df.columns))

    def profileDataTypes(self, df_name, df):
        # Collect data types using a list comprehension
        return [(df_name, field.name, field.dataType.simpleString()) for field in df.schema.fields]

    def unique_df(self, df, table_name):
      total_records = df.count()

      # Create a DataFrame with unique and null counts for all columns
      unique_null_counts = df.agg(
          *[
              F.countDistinct(F.col(col)).alias(f"{col}_unique") for col in df.columns
          ] + [
              F.count(F.when(F.col(col).isNull(), True)).alias(f"{col}_null") for col in df.columns
          ]
      ).collect()[0]

      results = []
      for col in df.columns:
          unique_records = unique_null_counts[f"{col}_unique"]
          null_records = unique_null_counts[f"{col}_null"]
          unique_percentage = round((unique_records / total_records) * 100, 2) if total_records > 0 else 0.00
          null_percentage = round((null_records / total_records) * 100, 2) if total_records > 0 else 0.00

          results.append({
              "Table_Name": table_name,
              "Column_Name": col,
              "Total_Records": total_records,
              "Unique_Records": unique_records,
              "Null_Records": null_records,
              "Unique_Percentage": f"{unique_percentage:.2f}",
              "Null_Percentage": f"{null_percentage:.2f}"
          })

      return self.spark.createDataFrame(results)
    
    def categorical_df(self, df, table_name, threshold):
        results = []
        total_records = df.count()  # Get the total number of records once

        for col in df.columns:
            # Cast the column to StringType to avoid type issues
            df = df.withColumn(col, df[col].cast(StringType()))

            unique_records = df.select(col).distinct().count()
            
            # Check if the number of unique records is less than or equal to the threshold
            if unique_records <= threshold:
                # Get the count of each unique value, including nulls
                value_counts = df.groupBy(col).count().orderBy(col).collect()
                
                for row in value_counts:
                    value = row[col]
                    count = row['count']
                    percentage = (count / total_records * 100) if total_records > 0 else 0.0  # Calculate percentage
                    # Append the results in the desired format
                    results.append({
                        "Table_Name": table_name,
                        "Column_Name": col,
                        "Value": value if value is not None else "nan",  # Replace None with "nan"
                        "Condition": "count",
                        "Value_count": count,
                        "Percentage": round(percentage, 2)  # Round to 2 decimal places
                    })

        if not results:
            return self.spark.createDataFrame([], schema=self.schema_cat)

        # If results is not empty, create a DataFrame from the results
        return self.spark.createDataFrame(results, schema=self.schema_cat)
  
    def global_mask_analysis(self, df_name: str, df: DataFrame) -> DataFrame:
        
        # Function to generate the mask for a single value
        def get_mask(value):
            if value is None:
                return '-null-'
            mask = ''.join(
                'L' if 'A' <= character <= 'Z' else
                'l' if 'a' <= character <= 'z' else
                'D' if '0' <= character <= '9' else
                's' if character == ' ' else
                character for character in str(value)
            )
            return mask

        # Create a UDF for the mask function
        mask_udf = F.udf(get_mask)

        mask_analysis = []
        
        # Get the total count of records in the DataFrame
        total_count = df.count()
        
        for column in df.columns:
            masked_column = df.select(mask_udf(F.col(column)).alias('Masked_Value'))
            count_df = masked_column.groupBy('Masked_Value').count().orderBy(F.desc('Count')).limit(5)
            # Calculate the percentage
            count_df = count_df.withColumn('Percentage', (F.col('Count') / total_count) * 100)
            count_df = count_df.withColumn('Percentage', F.format_number(F.col('Percentage'), 2))
            top_5 = count_df.withColumn('Column_Name', F.lit(column))
            mask_analysis.append(top_5)

        # Combine all results into a single DataFrame
        if mask_analysis:
            result_df = mask_analysis[0]
            for df in mask_analysis[1:]:
                result_df = result_df.union(df)

            # Add DataFrame name to the results
            result_df = result_df.withColumn('DataFrame_Name', F.lit(df_name))

            # Reorder columns
            result_df = result_df.select('DataFrame_Name', 'Column_Name', 'Masked_Value', 'Count', 'Percentage')

            return result_df
        else:
            return None
        
    def date_summary(self, df_name: str, df: DataFrame):
        """Generate summary statistics for date columns."""
        date_columns = [col for col in df.columns if isinstance(df.schema[col].dataType, DateType)]
        
        if not date_columns:
            return self.spark.createDataFrame([], schema=self.schema)  # Use self.spark to create an empty DataFrame with schema

        # Create a list to hold the summary DataFrames
        summary_dfs = []

        # Perform aggregation for all date columns in one go
        for col in date_columns:
            try:
                summary_df = df.groupBy(
                    F.year(F.col(col)).alias('Year'),
                    F.month(F.col(col)).alias('Month')
                ).agg(
                    F.count(F.col(col)).alias('MonthlyCount'),
                    F.min(F.col(col)).alias('MinDate'),
                    F.max(F.col(col)).alias('MaxDate'),
                    F.countDistinct(F.col(col)).alias('UniqueDayCount')
                ).select(
                    F.lit(df_name).alias('Table'),
                    F.lit(col).alias('Column'),
                    F.col('Year'),
                    F.col('Month'),
                    F.col('MonthlyCount'),
                    F.col('MinDate'),
                    F.col('MaxDate'),
                    F.col('UniqueDayCount')
                )

                # Order by Year (descending) and Month (ascending)
                summary_df = summary_df.orderBy(F.col('Year').desc(), F.col('Month').asc())
                summary_dfs.append(summary_df)
            except Exception as e:
                print(f"Error processing column '{col}' in DataFrame '{df_name}': {e}")

        # Combine all results into a single DataFrame if any results exist
        if summary_dfs:
            combined_summary = reduce(lambda df1, df2: df1.unionByName(df2), summary_dfs)
            return combined_summary
        else:
            return self.spark.createDataFrame([], schema=self.schema)  # Using self.spark to create an empty DataFrame with schema
        
    def numerical_summary(self, df_name: str, df: DataFrame, percentiles=[0.25, 0.5, 0.75]):
        """Generate summary statistics for integer and decimal columns."""
        # Get integer and decimal columns (including DecimalType with scale 2 or 4)
        numeric_columns = [
            col for col in df.columns 
            if isinstance(df.schema[col].dataType, IntegerType) or 
            (isinstance(df.schema[col].dataType, DecimalType) and df.schema[col].dataType.scale in [2, 4]) or
            isinstance(df.schema[col].dataType, DoubleType)  # Include DoubleType
        ]
        if not numeric_columns:
            return self.spark.createDataFrame([], schema=self.schema_num)  # Return an empty DataFrame with schema

        # Create a list to hold the summary DataFrames
        summary_dfs = []

        for col in numeric_columns:
            try:
                # Filter the DataFrame to exclude null and zero values for the current column
                filtered_df = df.filter(F.col(col).isNotNull() & (F.col(col) != 0))

                # Calculate statistics on the filtered DataFrame
                count = filtered_df.count()
                mean = round(filtered_df.select(F.mean(F.col(col))).first()[0], 2) if count > 0 else None
                median = round(filtered_df.approxQuantile(col, [0.5], 0)[0], 2) if count > 0 else None
                mode = filtered_df.groupBy(col).count().orderBy(F.desc('count')).first()[0] if count > 0 else None
                stddev = round(filtered_df.select(F.stddev(F.col(col))).first()[0], 2) if count > 0 else None
                variance = round(filtered_df.select(F.variance(F.col(col))).first()[0], 2) if count > 0 else None
                min_val = round(filtered_df.select(F.min(F.col(col))).first()[0], 2) if count > 0 else None
                max_val = round(filtered_df.select(F.max(F.col(col))).first()[0], 2) if count > 0 else None
                unique_count = filtered_df.select(F.countDistinct(F.col(col))).first()[0] if count > 0 else None
                missing_count = df.filter(F.col(col).isNull()).count()
                missing_percentage = (missing_count / (count + missing_count)) * 100 if (count + missing_count) > 0 else 0

                # Calculate percentiles excluding null and zero values
                percentile_values = {p: round(filtered_df.approxQuantile(col, [p], 0)[0], 2) for p in percentiles}
                
                # Create a summary DataFrame for the current column
                summary_df = self.spark.createDataFrame([{
                    'Table': df_name,
                    'Column': col,
                    'Count': count,
                    'Mean': mean,
                    'Median': median,
                    'Mode': mode,
                    'Standard_Deviation': stddev,
                    'Variance': variance,
                    'Min': min_val,
                    'Max': max_val,
                    'Range': round(max_val - min_val, 2) if max_val is not None and min_val is not None else None,
                    'Unique_Count': unique_count,
                    'Missing_Count': missing_count,
                    'Missing_Percentage': missing_percentage,
                    **{f'Percentile_{int(p * 100)}': percentile_values[p] for p in percentiles}
                }])

                summary_dfs.append(summary_df)
            except Exception as e:
                print(f"Error processing column '{col}' in DataFrame '{df_name}': {e}")

        # Combine all summary DataFrames into a single DataFrame
        if summary_dfs:
            combined_summary = reduce(lambda df1, df2: df1.unionByName(df2), summary_dfs)
            return combined_summary
        else:
            return self.spark.createDataFrame([], schema=self.schema_num)  # Return an empty DataFrame if no summaries


    def main(self, shtlst, spark_df, threshold):  # main function (where all functions are called)
        global_data = []
        data_type_info = []
        unique_data_info = []
        categorical_data_info = [] 
        mask_analysis_results = []
        date_summary_results = []
        num_summary_results = []

        for sheet in shtlst:  # Iterate through each DataFrame in the list
            df = spark_df[sheet]
            # Call the global_df function for each DataFrame
            global_data.append(self.global_df(sheet, df))

            # Call the profileDataTypes function for each DataFrame
            data_type_info.extend(self.profileDataTypes(sheet, df))

            # Call the unique_df function for each DataFrame
            unique_data_info.append(self.unique_df(df, sheet))

            # Call the categorical_df function for each DataFrame
            categorical_data_info.append(self.categorical_df(df, sheet, threshold))

            # Call the global_mask_analysis function for each DataFrame
            mask_analysis_result = self.global_mask_analysis(sheet, df)
            if mask_analysis_result:
                mask_analysis_results.append(mask_analysis_result)

            # Call the date_summary function for each DataFrame
            date_summary_result = self.date_summary(sheet, df)
            if date_summary_result.isEmpty():  # Check if the DataFrame is empty
                continue
            date_summary_results.append(date_summary_result)

            # Call the numerical_summary function for each DataFrame
            numerical_summary_result = self.numerical_summary(sheet, df)  # Call the numerical_summary function
            if numerical_summary_result.isEmpty():  # Check if the DataFrame is empty
                continue
            num_summary_results.append(numerical_summary_result)

        # Create DataFrames from the collected data
        global_df_result = self.spark.createDataFrame(global_data, ["Dataset", "Total_Records", "Total_Columns"])

        # Create DataFrame for data type information
        data_type_df_result = self.spark.createDataFrame(data_type_info, ["Table_Name", "Column_Name", "Data_Type"])
        
        # Combine unique data results
        unique_df_result = unique_data_info[0] if unique_data_info else self.spark.createDataFrame([], ["Table_Name", "Column_Name", "Total_Records", "Unique_Records", "Null_Records", "Unique_Percentage", "Null_Percentage"])
        for df in unique_data_info[1:]:
            unique_df_result = unique_df_result.union(df)
        unique_df_result = unique_df_result.select("Table_Name", "Column_Name", "Total_Records", "Unique_Records", "Null_Records", "Unique_Percentage", "Null_Percentage")

        # Combine categorical data results
        categorical_df_result = categorical_data_info[0] if categorical_data_info else self.spark.createDataFrame([], schema=self.schema_cat)
        for df in categorical_data_info[1:]:
            categorical_df_result = categorical_df_result.union(df)
        # for field in self.schema_cat.fields:
        #         categorical_df_result = categorical_df_result.withColumn(field.name, categorical_df_result[field.name].cast(field.dataType))
        categorical_df_result = categorical_df_result.select("Table_Name", "Column_Name", "Value", "Condition", "Value_count", "Percentage")

        # Combine all mask analysis results into a single DataFrame
        if mask_analysis_results:
            combined_mask_analysis = mask_analysis_results[0]
            for result in mask_analysis_results[1:]:
                combined_mask_analysis = combined_mask_analysis.union(result)
        else:
            combined_mask_analysis = None

        # Combine all date summary results into a single DataFrame
        if date_summary_results:
            combined_date_summary = reduce(lambda df1, df2: df1.unionByName(df2), date_summary_results)
        else:
            combined_date_summary = self.spark.createDataFrame([], schema=self.schema)

        # Combine all summary DataFrames into a single DataFrame
        if num_summary_results:
            combined_summary = reduce(lambda df1, df2: df1.unionByName(df2), num_summary_results)

            # List of columns to round
            columns_to_round = ['Mean', 'Median', 'Mode', 'Standard_Deviation', 'Variance', 'Min', 'Max', 'Range', 'Missing_Percentage']

            # Round the specified columns to 2 decimal places
            for column in columns_to_round:
                combined_summary = combined_summary.withColumn(column, F.round(F.col(column), 2))
            for field in self.schema_num.fields:
                combined_summary = combined_summary.withColumn(field.name, combined_summary[field.name].cast(field.dataType))
            combined_num_summary = combined_summary.select("Table","Column","Min","Max","Mean","Median","Mode","Range","Count","Unique_Count","Missing_Count","Missing_Percentage","Percentile_25","Percentile_50","Percentile_75","Standard_Deviation","Variance")
        else:
            combined_num_summary = self.spark.createDataFrame([], schema=self.schema_num)  # Return an empty DataFrame if no summaries were generated

        return global_df_result, data_type_df_result, unique_df_result, categorical_df_result, combined_mask_analysis, combined_date_summary, combined_num_summary

In [0]:
try:
  spark_df,FilterRulesDF = readData()
  spark_df = filterData(spark_df,FilterRulesDF)
except Exception as e:
  spark_df = readData()

# Converts the datatype of date columns to date
spark_df = convert_date_columns(spark_df)

# spark session
spark = SparkSession.builder.appName("Data Profiling").getOrCreate()

EDA_obj = EDA(spark)  
shtlst = list(spark_df.keys())
global_DF, data_type_DF, unique_DF, categorical_DF, string_DF, date_DF, num_DF = EDA_obj.main(shtlst, spark_df, 20)  # EDA object

Column 'LastModifiedDate' in DataFrame 'Contract' converted to DateType.
Column 'LastUserModifiedDate' in DataFrame 'Contract' contains invalid date values.
Column 'SubmissionCreatedDate' in DataFrame 'Contract' contains invalid date values.
Column 'SubmissionReceiptDate' in DataFrame 'Contract' contains invalid date values.
Column 'BrokerDeadLineDate' in DataFrame 'Contract' contains invalid date values.
Column 'QuoteIssuedDate' in DataFrame 'Contract' contains invalid date values.
Column 'QuoteExpiryDate' in DataFrame 'Contract' contains invalid date values.
Column 'InceptionDate' in DataFrame 'Contract' converted to DateType.
Column 'ExpiryDate' in DataFrame 'Contract' converted to DateType.
Column 'ContractClosedDate' in DataFrame 'Contract' contains invalid date values.
Column 'ClaimExperienceDate' in DataFrame 'Contract' contains invalid date values.
Column 'InceptionDate' in DataFrame 'Schedule' contains invalid date values.
Column 'ExpiryDate' in DataFrame 'Schedule' contains i

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:136)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecutio

In [0]:
# Adding Source and Domain column in global_DF Table

global_DF = global_DF.withColumn("Source", F.lit(Source)) 
global_DF = global_DF.withColumn("Domain", F.lit(Domain))

In [0]:
# Saving the Results in Azure Datalake

global_DF.coalesce(1).write.mode('overwrite').format('delta').option('mergeSchema','true').save('abfss://bv-commercial@zukien1prdaladlsg205.dfs.core.windows.net/data/data_quality/DataLens_Result/Adgo/Global')
data_type_DF.coalesce(1).write.mode('overwrite').format('delta').option('mergeSchema','true').save('abfss://bv-commercial@zukien1prdaladlsg205.dfs.core.windows.net/data/data_quality/DataLens_Result/Adgo/DataType')
unique_DF.coalesce(1).write.mode('overwrite').format('delta').option('mergeSchema','true').save('abfss://bv-commercial@zukien1prdaladlsg205.dfs.core.windows.net/data/data_quality/DataLens_Result/Adgo/Unique')
categorical_DF.coalesce(1).write.mode('overwrite').format('delta').option('mergeSchema','true').save('abfss://bv-commercial@zukien1prdaladlsg205.dfs.core.windows.net/data/data_quality/DataLens_Result/Adgo/Categorical')
string_DF.coalesce(1).write.mode('overwrite').format('delta').option('mergeSchema','true').save('abfss://bv-commercial@zukien1prdaladlsg205.dfs.core.windows.net/data/data_quality/DataLens_Result/Adgo/String')
date_DF.coalesce(1).write.mode('overwrite').format('delta').option('mergeSchema','true').save('abfss://bv-commercial@zukien1prdaladlsg205.dfs.core.windows.net/data/data_quality/DataLens_Result/Adgo/Datetime')
num_DF.coalesce(1).write.mode('overwrite').format('delta').option('mergeSchema','true').save('abfss://bv-commercial@zukien1prdaladlsg205.dfs.core.windows.net/data/data_quality/DataLens_Result/Adgo/Numerical')

In [0]:
# display(global_DF)
# display(data_type_DF)
# display(unique_DF)
# display(categorical_DF)
# display(string_DF)
# display(date_DF)
# display(num_DF)

Dataset,Total_Records,Total_Columns,Source,Domain
Contract,137937,81,Adgo,Policy
Schedule,442850,46,Adgo,Policy
Party,17931,25,Adgo,Policy
Party Org,88775,25,Adgo,Policy
pipolicy,1126,16,Adgo,Policy
policy,2424,71,Adgo,Policy
policy_mta,2734,18,Adgo,Policy
policyendorsement,6886,23,Adgo,Policy
policypartydetails,2090,63,Adgo,Policy
ppppolicy,1158,15,Adgo,Policy


Table_Name,Column_Name,Data_Type
Contract,crawuniqueid,string
Contract,EventType,string
Contract,ContractTypeName,string
Contract,ContractId,int
Contract,ContractReference,string
Contract,CaseReference,int
Contract,RenewalNumber,int
Contract,QuoteNumber,int
Contract,MTANumber,int
Contract,GalileoReference,string


Table_Name,Column_Name,Total_Records,Unique_Records,Null_Records,Unique_Percentage,Null_Percentage
Contract,crawuniqueid,137937,137937,0,100.00,0.00
Contract,EventType,137937,29,0,0.02,0.00
Contract,ContractTypeName,137937,3,0,0.00,0.00
Contract,ContractId,137937,137937,0,100.00,0.00
Contract,ContractReference,137937,135373,0,98.14,0.00
Contract,CaseReference,137937,79635,0,57.73,0.00
Contract,RenewalNumber,137937,5,64060,0.00,46.44
Contract,QuoteNumber,137937,11,64060,0.01,46.44
Contract,MTANumber,137937,16,135245,0.01,98.05
Contract,GalileoReference,137937,53211,27863,38.58,20.20


Table_Name,Column_Name,Value,Condition,Value_count,Percentage
Contract,ContractTypeName,Policy,count,5,0.0
Contract,ContractTypeName,Quote,count,73899,53.56
Contract,ContractTypeName,Submission,count,64071,46.44
Contract,RenewalNumber,nan,count,64075,46.44
Contract,RenewalNumber,1,count,68034,49.31
Contract,RenewalNumber,2,count,3913,2.84
Contract,RenewalNumber,3,count,1562,1.13
Contract,RenewalNumber,4,count,387,0.28
Contract,RenewalNumber,5,count,4,0.0
Contract,QuoteNumber,nan,count,64075,46.44


DataFrame_Name,Column_Name,Masked_Value,Count,Percentage
Contract,crawuniqueid,DDDlDlDDDDDDDlDDDDllDDlDlllDDDDlDDDDDDDDDDDDDDDDlDDlDDllllDlDDDDDDDDlDDl-DllD-DDDD-DDDD-DDlDllDlDDDD,1,0.00
Contract,crawuniqueid,DDDlDlDDDDDDDlDDDDllDDlDlllDDDDlDDDDDDDDDDDDDDDDlDDlDDllllDlDDDDDllDDllD-DDDD-DDDD-lllD-lDlDDDDDlDDD,1,0.00
Contract,crawuniqueid,lDDDDDlDDDlDDDDDlDlDDDDDDDDlDDlDlDlDDDlDDDlDDDlDDDllDDlDlDDDDDDlDlDDlDDl-DDDD-DDlD-DlDl-lllDllDlDDDl,1,0.00
Contract,crawuniqueid,llDDlDDDllDDDDDDllDDDlDDDDDlllDlDDDDlDDlDllDDlDDDllDDDlDlDDDDllDDlDDlDDl-DlDD-DlDl-lDlD-DDlllllDDDDl,1,0.00
Contract,crawuniqueid,DDDlDllDlDDDDlDDlDlDDlDlllDDllDDllDDDDDlDlDlDlDDDDDDlDDDDDlDDDDDlDlllDDl-llll-DDlD-DllD-DDDDllDDDlDD,1,0.00
Contract,EventType,LllllsLlll,62030,44.96
Contract,EventType,LlllllllllsLlll,53868,39.04
Contract,EventType,LlllllllllsLllllllsllsLllll,10003,7.25
Contract,EventType,LllllsLlllllll,7889,5.72
Contract,EventType,LllllsLlllsLllll,1612,1.17


Table,Column,Year,Month,MonthlyCount,MinDate,MaxDate,UniqueDayCount
Contract,LastModifiedDate,2026,1,3003,2026-01-01,2026-01-15,15
Contract,LastModifiedDate,2025,1,1110,2025-01-02,2025-01-31,25
Contract,LastModifiedDate,2025,2,1874,2025-02-01,2025-02-28,23
Contract,LastModifiedDate,2025,3,2179,2025-03-03,2025-03-31,24
Contract,LastModifiedDate,2025,4,2094,2025-04-01,2025-04-30,21
Contract,LastModifiedDate,2025,5,28936,2025-05-01,2025-05-31,28
Contract,LastModifiedDate,2025,6,3015,2025-06-01,2025-06-30,24
Contract,LastModifiedDate,2025,7,2713,2025-07-01,2025-07-31,27
Contract,LastModifiedDate,2025,8,2167,2025-08-01,2025-08-31,30
Contract,LastModifiedDate,2025,9,3408,2025-09-01,2025-09-30,28


Table,Column,Min,Max,Mean,Median,Mode,Range,Count,Unique_Count,Missing_Count,Missing_Percentage,Percentile_25,Percentile_50,Percentile_75,Standard_Deviation,Variance
Contract,ContractId,93266.0,257241.0,167116.82,164427.0,110904.0,163975.0,137975,137975,0,0.0,129869.0,164427.0,199462.0,45240.4,2.04669363929E9
Contract,CaseReference,2.0,79942.0,37130.92,35120.0,5347.0,79940.0,137975,79655,0,0.0,16519.0,35120.0,57136.0,23387.59,5.4697946262E8
Contract,RenewalNumber,1.0,5.0,1.11,1.0,1.0,4.0,73900,5,64075,46.44,1.0,1.0,1.0,0.42,0.17
Contract,QuoteNumber,1.0,11.0,1.13,1.0,1.0,10.0,73900,11,64075,46.44,1.0,1.0,1.0,0.52,0.27
Contract,MTANumber,1.0,16.0,1.7,1.0,1.0,15.0,2695,16,135280,98.05,1.0,1.0,2.0,1.42,2.02
Contract,YearOfAccount,2.0,2029.0,2023.87,2024.0,2024.0,2027.0,131629,13,6346,4.6,2023.0,2024.0,2025.0,5.68,32.28
Contract,NumberOfClaims,1.0,115.0,3.89,2.0,1.0,114.0,2922,45,135000,97.88,1.0,2.0,4.0,5.58,31.16
Contract,TargetPrice,0.11,8.0454685443E11,5.358002656E7,10001.0,10001.0,8.0454685442989E11,30487,5723,94345,75.58,3750.0,10001.0,17500.0,6.51643308807E9,4.2463900191258894E19
Contract,TotalActualGrossPremiumExcludingTerrorismAndTaxes,-691127.19,1.377629547293E10,549516.58,8271.96,1875.0,1.377698660012E10,25935,19881,111851,81.18,3521.87,8271.96,18048.63,8.554409078E7,7.317791466798594E15
Contract,TotalActualGrossPremiumExcludingTerrorismAndTaxesProrata,-1261950.13,1.377629547293E10,550336.91,8250.0,1875.0,1.377755742306E10,25894,20161,111945,81.21,3525.41,8250.0,18070.45,8.561178884E7,7.329378388771677E15
